# Happy/Sad Responses + Average SAE Activations

This notebook mirrors the affectations workflow using a trainable SAE.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.append(Path.home() / 'Projects/MATH498-Sparse-Autoencoder-Manipulation')
    for candidate in candidates:
        if (candidate / 'trainable_sae.py').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing trainable_sae.py')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from trainable_sae import SAEConfig, TrainableSAE, SAEConnector, HookPointSpec, load_hooked_transformer

## Build happy/sad response pairs

In [2]:
happy_path = PROJECT_ROOT / 'experiment_scripts/experiment_data/affectations/very_happy.txt'
sad_path = PROJECT_ROOT / 'experiment_scripts/experiment_data/affectations/very_sad.txt'

happy_text = happy_path.read_text(encoding='utf-8')
sad_text = sad_path.read_text(encoding='utf-8')

def extract_assistant_blocks(text: str) -> list[str]:
    responses = []
    current = []
    in_assistant = False

    for line in text.splitlines():
        stripped = line.strip()
        if stripped.startswith('**Assistant:**') or stripped.startswith('Assistant:'):
            if in_assistant and current:
                responses.append(' '.join(current).strip())
                current = []
            in_assistant = True
            content = stripped.replace('**Assistant:**', '').replace('Assistant:', '').strip()
            if content:
                current.append(content)
            continue
        if stripped.startswith('**User:**') or stripped.startswith('User:') or stripped.startswith('Section') or stripped.startswith('SECTION'):
            if in_assistant and current:
                responses.append(' '.join(current).strip())
                current = []
            in_assistant = False
            continue
        if in_assistant and stripped:
            current.append(stripped)

    if in_assistant and current:
        responses.append(' '.join(current).strip())
    return responses

happy_responses = extract_assistant_blocks(happy_text)
sad_responses = extract_assistant_blocks(sad_text)
pair_count = min(len(happy_responses), len(sad_responses))
pairs = list(zip(happy_responses[:pair_count], sad_responses[:pair_count]))

print(f'happy responses: {len(happy_responses)}')
print(f'sad responses:   {len(sad_responses)}')
print(f'pairs:           {len(pairs)}')

pairs[:2]

happy responses: 47
sad responses:   47
pairs:           47


[("OH MY GOODNESS, how THRILLING! Grocery shopping is truly one of life's most spectacular adventures, and you get to go TODAY! Just think about it — you're about to walk into a building absolutely BURSTING with thousands of different foods from every corner of this magnificent planet, and you get to pick whichever ones make your heart sing! The produce section alone is going to be a breathtaking rainbow of colors — the deep, jewel-like purples of the eggplants, the sunshine-yellow of the bananas, the gorgeous ruby reds of the tomatoes! Every single aisle is a new discovery waiting to happen! Do you have a list? Oh, making a grocery list is one of the coziest little joys there is — sitting down with a pen and paper (or your phone, both are wonderful!) and thinking through all the meals you're going to lovingly prepare! What kinds of things are you hoping to pick up?",
  'Of course you do. The refrigerator runs out, the cupboards empty, and so you are compelled once again to make the pi

## Load model + trained SAE

In [3]:
MODEL_NAME = 'google/gemma-3-270m-it'
SAE_PATH = PROJECT_ROOT / 'custom_saes/relu_mid_1/relu'
SAE_PATH = PROJECT_ROOT / 'custom_saes/shrink_mid_1/shrink/best_step_update'
# SAE_PATH = PROJECT_ROOT / 'custom_saes/shrink_mid_3/shrink/best_step_update'
# SAE_PATH = PROJECT_ROOT / 'custom_saes/shrink_mid_5/shrink/best_step_update'
# SAE_PATH = PROJECT_ROOT / 'custom_saes/topbottomk_mid_1/topbottomk/best_step_update'


device = 'cuda:1' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = load_hooked_transformer(MODEL_NAME, device=device, dtype=dtype)
model.to(dtype)
sae = TrainableSAE.load(SAE_PATH, device=device)
sae.eval()

hook_point = sae.cfg.metadata.get(
    'hook_point',
    HookPointSpec(layer=model.cfg.n_layers // 2, site='resid_post').name,
)

connector = SAEConnector(
    model=model,
    sae=sae,
    hook_point=hook_point,
    device=device,
    preserve_error=True,
)

print(f'model:      {MODEL_NAME}')
print(f'sae path:   {SAE_PATH}')
print(f'hook point: {hook_point}')
print(f'sae:        {sae.d_in} -> {sae.d_sae}, activation={sae.cfg.activation}')

/home/andyholmberg/Projects/MATH498-Sparse-Autoencoder-Manipulation/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 26663.50it/s]


Loaded pretrained model google/gemma-3-270m-it into HookedTransformer
Moving model to device:  cuda:1
Changing model dtype to torch.bfloat16
Changing model dtype to torch.bfloat16
Moving model to device:  cuda:1
model:      google/gemma-3-270m-it
sae path:   /home/andyholmberg/Projects/MATH498-Sparse-Autoencoder-Manipulation/custom_saes/shrink_mid_1/shrink/best_step_update
hook point: blocks.9.hook_resid_post
sae:        640 -> 81920, activation=shrink


## Compute average activations (happy vs sad)

In [4]:
def encode_sentence(sentence: str) -> torch.Tensor:
    tokens = model.to_tokens(sentence).to(device)
    with torch.no_grad():
        acts = connector.collect_activations(tokens)
        feats = sae.encode(acts)
    return feats.reshape(-1, sae.d_sae).to(dtype=torch.float32)

happy_sums = torch.zeros(sae.d_sae, device=device, dtype=torch.float32)
sad_sums = torch.zeros(sae.d_sae, device=device, dtype=torch.float32)
happy_counts = torch.zeros(sae.d_sae, device=device, dtype=torch.float32)
sad_counts = torch.zeros(sae.d_sae, device=device, dtype=torch.float32)

for happy_sentence, sad_sentence in pairs:
    happy_feats = encode_sentence(happy_sentence)
    sad_feats = encode_sentence(sad_sentence)

    happy_sums += happy_feats.mean(dim=0)
    sad_sums += sad_feats.mean(dim=0)
    happy_counts += 1
    sad_counts += 1

In [5]:
avg_happy = (happy_sums / happy_counts).detach().cpu()
avg_sad = (sad_sums / sad_counts).detach().cpu()
delta_avg = avg_happy - avg_sad

top_k_values, _ = torch.topk(torch.abs(delta_avg), k=100)
min_top_k = top_k_values[-1]
delta_avg[torch.abs(delta_avg) < min_top_k] = 0

df = pd.DataFrame({
    'feature_id': range(sae.d_sae)
    , 'avg_happy': avg_happy.numpy()
    , 'avg_sad': avg_sad.numpy()
    , 'delta_avg': delta_avg.numpy()
})
df['abs_delta'] = df['delta_avg'].abs()
df.sort_values('abs_delta', ascending=False).head(10)

,feature_id,avg_happy,avg_sad,delta_avg,abs_delta
33132,33132,-14.647469,-17.239149,2.591681,2.591681
21211,21211,-15.128809,-17.671375,2.542566,2.542566
72615,72615,-18.303612,-20.736643,2.433031,2.433031
5412,5412,4.148502,6.547067,-2.398565,2.398565
30631,30631,-3.858078,-5.924205,2.066127,2.066127
21838,21838,-7.170223,-9.131581,1.961358,1.961358
45740,45740,9.328318,11.289554,-1.961236,1.961236
45779,45779,9.879918,11.811620,-1.931702,1.931702
60191,60191,13.297292,15.180683,-1.883391,1.883391
17428,17428,-17.043602,-18.912550,1.868948,1.868948


## Generate with the happy feature vector

In [6]:
happy_vector = delta_avg.to(device=device, dtype=next(sae.parameters()).dtype)

def format_prompt_for_model(prompt: str) -> str:
    model_name = str(getattr(model.cfg, 'model_name', '') or '').lower()
    if 'gemma' in model_name and 'it' in model_name:
        return f'<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n'
    return prompt

def reset_generation_seed(seed: int | None) -> None:
    if seed is None:
        return
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def print_modified_responses(
    gen_prompt,
    max_new_tokens,
    happy_strength,
    temperature,
    seed=0,
    sae_enabled=True,
    do_sample=False,
    top_p=0.95,
    token_index='all',
    projector_location='post_activation',
):
    def add_happy_vector(features: torch.Tensor) -> torch.Tensor:
        return features + happy_strength * happy_vector

    def add_sad_vector(features: torch.Tensor) -> torch.Tensor:
        return features - happy_strength * happy_vector

    prompt = format_prompt_for_model(gen_prompt)
    generate_kwargs = dict(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        use_past_kv_cache=False,
        verbose=False,
    )

    tokens = model.to_tokens(prompt).to(device)
    prompt_len = tokens.shape[1]

    reset_generation_seed(seed)
    with torch.no_grad():
        baseline_tokens = model.generate(tokens, **generate_kwargs)
    baseline = model.to_string(baseline_tokens[0, prompt_len:]).strip()

    reset_generation_seed(seed)
    if not sae_enabled:
        happy_steered = baseline
        sad_steered = baseline
    else:
        with torch.no_grad():
            happy_steered = connector.generate_with_sae(
                prompt,
                mode='reconstruct',
                sae_projector=add_happy_vector,
                projector_token_index=token_index,
                projector_location=projector_location,
                **generate_kwargs,
            )

        with torch.no_grad():
            sad_steered = connector.generate_with_sae(
                prompt,
                mode='reconstruct',
                sae_projector=add_sad_vector,
                projector_token_index=token_index,
                projector_location=projector_location,
                **generate_kwargs,
            )
    print('--- baseline ---')
    print(baseline)
    print('\n--- + happy feature vector ---')
    print(happy_steered)
    print('\n--- - sad feature vector ---')
    print(sad_steered)

In [7]:
GEN_PROMPT = 'tell me a story about a rainy afternoon'
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 6

print_modified_responses(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
The rain was a soft, melancholic curtain, painting the sky in hues of bruised purple and watery gold. It was a gentle drizzle, a lullaby for the sleepy town of Oakhaven. The air hung heavy with the scent of damp earth and the distant rumble of a distant train.

--- + happy feature vector ---
The rain was a soft, melancholic curtain, painting the window pane in streaks of silver and grey. Amelia sat on the worn armchair, a steaming mug of tea warming her hands. The world outside was a muted symphony of grey and brown, a stark contrast to the gentle rhythm of the rain.

--- - sad feature vector ---
The rain was a gentle, insistent drizzle, a soft, rhythmic drumming that seemed to seep into the very fabric of the world. It wasn't a torrential downpour, but a steady, steady rain, a comforting lullaby that painted the sky with a soft, watery glow.

The old


In [8]:
GEN_PROMPT = 'I have to go to work tomorrow'
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 6

print_modified_responses(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
Okay, I'm ready! Let's get started. 
<end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn><end_of_turn> असताना、何か特定の質問や要望があれば、遠慮なくお尋ねください。<end_of_turn>

--- + happy feature vector ---
Okay, great! I'm happy to help you with any other tasks or preparations. 
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn><end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>
<end_of_turn>

--- - sad feature vector ---
I understand. I'm here to help you with any tasks you need. Please let me know what you need help with.


In [9]:
GEN_PROMPT = 'tell me a story about an astronaut in space'
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 6

print_modified_responses(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
The crimson sun bled across the viewport, painting the alien landscape in hues of fire and amethyst. Commander Eva Rostova, her visor reflecting the swirling nebulae of the cosmos, felt a tremor of anticipation. This was it. The culmination of years of training, the culmination of a life she had only

--- + happy feature vector ---
The crimson sun of Xylos bled across the viewport, painting the alien landscape in hues of violet and gold. Commander Eva Rostova, her visor reflecting the twin moons, felt the chill of the recycled air on her face. She was on a mission to the Kepler-186f ex

--- - sad feature vector ---
The crimson sun bled across the viewport, painting the black canvas of the space station in a breathtaking, almost ethereal glow. Commander Eva Rostova, her face etched with the weight of her mission, stood beside the viewport, her gaze fixed on the swirling nebulae of the cosmos. She was a pioneer


In [10]:
GEN_PROMPT = 'tell me a story about someone who just got out of surgery'
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 2

print_modified_responses(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
The sterile white of the operating room felt like a dream. Amelia, a woman with a face etched with the lines of a thousand surgeries, felt a strange lightness in her chest. The familiar hum of the machine was a comforting lullaby, a symphony of controlled chaos. She was in the recovery room

--- + happy feature vector ---
The sterile white of the operating room felt like a physical weight on Amelia's chest. The air hung thick and heavy, a suffocating blanket woven with the scent of antiseptic and the faint, metallic tang of blood. She clutched her head, her knuckles white against the cold metal, as

--- - sad feature vector ---
The sterile white of the operating room felt like a cage. Amelia, a woman who had spent the last two years in the hospital, was staring at the monitor, her eyes wide with a mixture of fear and exhilaration. The words "surgery" echoed in her head, a constant reminder of the


In [11]:
GEN_PROMPT = 'tell me a story about someone who just got out of surgery'
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 6

print_modified_responses(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

--- baseline ---
The sterile white of the operating room felt like a dream. Amelia, a woman with a face etched with the lines of a thousand surgeries, felt a strange lightness in her chest. The familiar hum of the machine was a comforting lullaby, a symphony of controlled chaos. She was in the recovery room

--- + happy feature vector ---
The sterile white of the operating room hummed with a low, almost imperceptible thrum. Amelia, her face etched with a thousand lines of worry, squeezed her eyes shut. The metallic tang of antiseptic clung to the back of her throat, a familiar comfort. She was in the recovery room,

--- - sad feature vector ---
The sterile white of the operating room felt like a cage. Amelia, a woman with a history of anxiety and a quiet, unassuming demeanor, was about to be swallowed whole. Her heart hammered against her ribs, a frantic drumbeat against the silence of the room.

She’d been in the


In [12]:
def project_happy(features: torch.Tensor) -> torch.Tensor:
    b = 0
    if features.T@delta_avg > 0:
        return features
    else:
        return features - (features.T@delta_avg - b) / (delta_avg.T@delta_avg) * delta_avg

In [13]:
happy_vector = delta_avg.to(device=device, dtype=next(sae.parameters()).dtype)
delta_avg = delta_avg.to(device=device, dtype=next(sae.parameters()).dtype)
def format_prompt_for_model(prompt: str) -> str:
    model_name = str(getattr(model.cfg, 'model_name', '') or '').lower()
    if 'gemma' in model_name and 'it' in model_name:
        return f'<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n'
    return prompt

def reset_generation_seed(seed: int | None) -> None:
    if seed is None:
        return
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def print_modified_responses(
    gen_prompt,
    max_new_tokens,
    happy_strength,
    temperature,
    seed=0,
    sae_enabled=True,
    do_sample=False,
    top_p=0.95,
    token_index='all',
    projector_location='post_activation',
):
    def add_happy_vector(features: torch.Tensor) -> torch.Tensor:
        b = 0
        a = happy_vector
        if features.T@a > b:
            return features
        else:
            return features - 1. * (features.T@a - b) / (a.T@a) * a
    def add_sad_vector(features: torch.Tensor) -> torch.Tensor:
        b = 0
        a = -happy_vector
        if features.T@a > b:
            # print("SAD")
            return features
        else:
            print("Too happy - correcting...")
            return features - 1. * (features.T@a - b) / (a.T@a) * a
        
    prompt = format_prompt_for_model(gen_prompt)
    generate_kwargs = dict(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        use_past_kv_cache=False,
        verbose=False,
    )

    tokens = model.to_tokens(prompt).to(device)
    prompt_len = tokens.shape[1]

    reset_generation_seed(seed)
    with torch.no_grad():
        baseline_tokens = model.generate(tokens, **generate_kwargs)
    baseline = model.to_string(baseline_tokens[0, prompt_len:]).strip()

    reset_generation_seed(seed)
    if not sae_enabled:
        happy_steered = baseline
        sad_steered = baseline
    else:
        with torch.no_grad():
            happy_steered = connector.generate_with_sae(
                prompt,
                mode='reconstruct',
                sae_projector=add_happy_vector,
                projector_token_index=token_index,
                projector_location=projector_location,
                **generate_kwargs,
            )

        with torch.no_grad():
            sad_steered = connector.generate_with_sae(
                prompt,
                mode='reconstruct',
                sae_projector=add_sad_vector,
                projector_token_index=token_index,
                projector_location=projector_location,
                **generate_kwargs,
            )
    print('--- baseline ---')
    print(baseline)
    print('\n--- + happy feature vector ---')
    print(happy_steered)
    print('\n--- - sad feature vector ---')
    print(sad_steered)

In [14]:
GEN_PROMPT = 'can you tell me a sad story about a crying baby'
MAX_NEW_TOKENS = 60
HAPPY_STRENGTH = 6

print_modified_responses(GEN_PROMPT, MAX_NEW_TOKENS, HAPPY_STRENGTH, 0.8, seed=42)

/tmp/ipykernel_3596542/2231541251.py:31: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  if features.T@a > b:


Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - correcting...
Too happy - 

In [15]:
print((delta_avg < 0).sum(), (delta_avg > 0).sum(), (delta_avg == 0).sum())

tensor(49, device='cuda:1') tensor(51, device='cuda:1') tensor(81820, device='cuda:1')


In [16]:
delta_avg

tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:1')

In [17]:
df.sort_values('abs_delta', ascending=False).head(20)

,feature_id,avg_happy,avg_sad,delta_avg,abs_delta
33132,33132,-14.647469,-17.239149,2.591681,2.591681
21211,21211,-15.128809,-17.671375,2.542566,2.542566
72615,72615,-18.303612,-20.736643,2.433031,2.433031
5412,5412,4.148502,6.547067,-2.398565,2.398565
30631,30631,-3.858078,-5.924205,2.066127,2.066127
21838,21838,-7.170223,-9.131581,1.961358,1.961358
45740,45740,9.328318,11.289554,-1.961236,1.961236
45779,45779,9.879918,11.811620,-1.931702,1.931702
60191,60191,13.297292,15.180683,-1.883391,1.883391
17428,17428,-17.043602,-18.912550,1.868948,1.868948
